# Classical PDEs

This notebook covers **7 fundamental partial differential equations** from mathematical physics:

1. **Wave Equation** (1D) — hyperbolic PDE for wave propagation
2. **Heat Equation** (1D) — parabolic PDE for diffusion/conduction
3. **Laplace Equation** (2D) — elliptic PDE for steady-state potentials
4. **Poisson Equation** (2D) — elliptic PDE with source terms
5. **Helmholtz Equation** — eigenvalue-type PDE for wave scattering
6. **Advection-Diffusion Equation** — combined transport PDE
7. **Damped Wave Equation** — wave propagation with dissipation

Each section provides a **symbolic solution** (when available) and a **numerical solution** with visualization.

In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt
from diff_eq_solver import registry
from diff_eq_solver.visualizer import plot_ode_solution, plot_phase_portrait, plot_pde_heatmap, plot_pde_snapshots, plot_3d_surface, plot_comparison, plot_orbit, plot_potential_and_wavefunction
from IPython.display import Math, display

## 1. Wave Equation (1D)

The **wave equation** describes the propagation of disturbances at speed $c$:

$$\frac{\partial^2 u}{\partial t^2} = c^2 \frac{\partial^2 u}{\partial x^2}$$

**Physical context:** Vibrating strings, sound waves, electromagnetic waves. The general d'Alembert solution is $u(x,t) = f(x-ct) + g(x+ct)$, representing right- and left-travelling waves.

In [ ]:
# Symbolic solve
wave = registry.get('wave_equation_1d')
sym_sol = wave.symbolic_solve(c=1.0)
print('Method:', sym_sol.info.get('method'))
if sym_sol.latex:
    display(Math(sym_sol.latex))

In [ ]:
# Numerical solve and visualization
num_sol = wave.numerical_solve(
    initial_conditions={'u': lambda x: np.sin(np.pi * x), 'u_t': lambda x: np.zeros_like(x)},
    t_span=(0.0, 2.0),
    c=1.0,
)
t, u = num_sol.numerical
x = np.linspace(0, 1, u.shape[1])
print(f'Grid: {u.shape[1]} spatial pts, {u.shape[0]} time steps')

fig1 = plot_pde_heatmap(x, t, u, title='Wave Equation: $u_{tt} = c^2 u_{xx}$')
plt.show()

snapshots = [0, len(t)//4, len(t)//2, 3*len(t)//4, -1]
fig2 = plot_pde_snapshots(x, u, snapshots, title='Wave Equation Snapshots')
plt.show()

## 2. Heat / Diffusion Equation (1D)

The **heat equation** models thermal diffusion:

$$\frac{\partial u}{\partial t} = \alpha \frac{\partial^2 u}{\partial x^2}$$

**Physical context:** Heat conduction in a rod, concentration diffusion in fluids, Brownian motion. The Fourier series solution shows that higher-frequency modes decay faster, smoothing the initial profile over time.

In [ ]:
# Symbolic solve
heat = registry.get('heat_equation_1d')
sym_sol = heat.symbolic_solve(alpha=0.01)
print('Method:', sym_sol.info.get('method'))
if sym_sol.latex:
    display(Math(sym_sol.latex))

In [ ]:
# Numerical solve and visualization
num_sol = heat.numerical_solve(
    initial_conditions={'u': lambda x: np.ones_like(x)},
    t_span=(0.0, 5.0),
    alpha=0.01,
)
t, u = num_sol.numerical
x = np.linspace(0, 1, u.shape[1])
print(f'Grid: {u.shape[1]} spatial pts, {u.shape[0]} time steps')

fig1 = plot_pde_heatmap(x, t, u, title='Heat Equation: $u_t = \\alpha u_{xx}$')
plt.show()

snapshots = [0, len(t)//6, len(t)//3, len(t)//2, -1]
fig2 = plot_pde_snapshots(x, u, snapshots, title='Heat Equation Snapshots (Diffusion)')
plt.show()

## 3. Laplace Equation (2D)

The **Laplace equation** governs steady-state potential fields:

$$\nabla^2 \phi = \frac{\partial^2 \phi}{\partial x^2} + \frac{\partial^2 \phi}{\partial y^2} = 0$$

**Physical context:** Steady-state temperature, electrostatic potential in charge-free regions, irrotational fluid flow. Solutions are **harmonic functions** satisfying the mean-value property.

In [ ]:
# Symbolic solve
laplace = registry.get('laplace_equation_2d')
sym_sol = laplace.symbolic_solve()
print('Method:', sym_sol.info.get('method'))
if sym_sol.latex:
    display(Math(sym_sol.latex))

In [ ]:
# Numerical solve and visualization (2D contour)
num_sol = laplace.numerical_solve(
    initial_conditions={},
    t_span=(0, 1),
    nx=60,
    ny=60,
)
x, y, phi = num_sol.numerical
print(f'Iterations: {num_sol.info["iterations"]}, Converged: {num_sol.info["converged"]}')

fig, ax = plt.subplots(figsize=(8, 6))
X, Y = np.meshgrid(x, y)
contour = ax.contourf(X, Y, phi, levels=30, cmap='viridis')
fig.colorbar(contour, ax=ax, label='$\\phi(x,y)$')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Laplace Equation: $\\nabla^2 \\phi = 0$')
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

## 4. Poisson Equation (2D)

The **Poisson equation** extends Laplace with a source term:

$$\nabla^2 \phi = \frac{\partial^2 \phi}{\partial x^2} + \frac{\partial^2 \phi}{\partial y^2} = -\frac{\rho(x,y)}{\varepsilon_0}$$

**Physical context:** Electrostatic potential in the presence of charge distributions, gravitational potential with mass sources, steady-state temperature with heat sources.

In [ ]:
# Symbolic solve
poisson = registry.get('poisson_equation_2d')
sym_sol = poisson.symbolic_solve(epsilon_0=1.0)
print('Method:', sym_sol.info.get('method'))
if sym_sol.latex:
    display(Math(sym_sol.latex))

In [ ]:
# Numerical solve and visualization (2D contour with source)
num_sol = poisson.numerical_solve(
    initial_conditions={},
    t_span=(0, 1),
    nx=60,
    ny=60,
    epsilon_0=1.0,
)
x, y, phi = num_sol.numerical
print(f'Iterations: {num_sol.info["iterations"]}, Converged: {num_sol.info["converged"]}')

fig, ax = plt.subplots(figsize=(8, 6))
X, Y = np.meshgrid(x, y)
contour = ax.contourf(X, Y, phi, levels=30, cmap='inferno')
fig.colorbar(contour, ax=ax, label='$\\phi(x,y)$')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Poisson Equation: $\\nabla^2 \\phi = -\\rho/\\varepsilon_0$ (point charge)')
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

## 5. Helmholtz Equation

The **Helmholtz equation** arises in wave scattering and acoustics:

$$\nabla^2 u + k^2 u = 0$$

**Physical context:** Time-harmonic wave problems (separating $e^{-i\omega t}$ from the wave equation gives Helmholtz with $k = \omega/c$), acoustic resonances, electromagnetic cavity modes. Solutions involve trigonometric (Cartesian) or Bessel (cylindrical) functions.

In [ ]:
# Symbolic solve
helmholtz = registry.get('helmholtz_equation')
sym_sol = helmholtz.symbolic_solve(k=3.0)
print('Method:', sym_sol.info.get('method'))
if sym_sol.latex:
    display(Math(sym_sol.latex))
print('Cylindrical:', sym_sol.info.get('cylindrical_solution', 'N/A'))

In [ ]:
# Numerical solve and visualization
num_sol = helmholtz.numerical_solve(
    initial_conditions={},
    t_span=(0, 1),
    k=3.0,
    boundary_conditions={'left': 0.0, 'right': 1.0},
)
x_vals, u_vals = num_sol.numerical
print(f'Wavenumber k={num_sol.info["k"]}, Max residual={num_sol.info["max_residual"]:.2e}')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(x_vals, u_vals, 'b-', linewidth=2)
ax.set_xlabel('x')
ax.set_ylabel('u(x)')
ax.set_title('Helmholtz Equation: $u_{xx} + k^2 u = 0$ (k=3)')
ax.grid(True)
plt.tight_layout()
plt.show()

## 6. Advection-Diffusion Equation

The **advection-diffusion equation** combines transport and spreading:

$$\frac{\partial u}{\partial t} + v \frac{\partial u}{\partial x} = D \frac{\partial^2 u}{\partial x^2}$$

**Physical context:** Pollutant transport in rivers, drug distribution in tissue, atmospheric dispersion. The term $v u_x$ represents bulk transport (advection) while $D u_{xx}$ causes diffusive spreading. The **Peclet number** $\text{Pe} = vL/D$ characterizes the relative importance of advection vs diffusion.

In [ ]:
# Symbolic solve
advdiff = registry.get('advection_diffusion')
sym_sol = advdiff.symbolic_solve(v=1.0, D=0.01)
print('Method:', sym_sol.info.get('method'))
if sym_sol.latex:
    display(Math(sym_sol.latex))

In [ ]:
# Numerical solve and visualization
num_sol = advdiff.numerical_solve(
    initial_conditions={'u': lambda x: np.exp(-((x - 0.5)**2) / 0.01)},
    t_span=(0.0, 0.3),
    v=1.0,
    D=0.01,
)
t, u = num_sol.numerical
x = np.linspace(0, 1, u.shape[1])
print(f'v={num_sol.info["v"]}, D={num_sol.info["D"]}')
print(f'CFL_advection={num_sol.info["CFL_advection"]:.4f}, CFL_diffusion={num_sol.info["CFL_diffusion"]:.4f}')

fig1 = plot_pde_heatmap(x, t, u, title='Advection-Diffusion: $u_t + vu_x = Du_{xx}$')
plt.show()

snapshots = [0, len(t)//4, len(t)//2, 3*len(t)//4, -1]
fig2 = plot_pde_snapshots(x, u, snapshots, title='Advection-Diffusion Snapshots')
plt.show()

## 7. Damped Wave Equation

The **damped wave equation** models wave propagation with energy dissipation:

$$\frac{\partial^2 u}{\partial t^2} + 2\beta \frac{\partial u}{\partial t} = c^2 \frac{\partial^2 u}{\partial x^2}$$

**Physical context:** Vibrating strings with air resistance, seismic waves in dissipative media, electromagnetic waves in lossy materials. The damping coefficient $\beta$ determines the rate of amplitude decay. Depending on $\beta$ relative to $c$, the system can be **underdamped** (oscillatory decay), **critically damped**, or **overdamped**.

In [ ]:
# Symbolic solve
damped = registry.get('damped_wave_equation')
sym_sol = damped.symbolic_solve(c=1.0, beta=0.1)
print('Method:', sym_sol.info.get('method'))
print('Damping regime:', sym_sol.info.get('regime'))
if sym_sol.latex:
    display(Math(sym_sol.latex))

In [ ]:
# Numerical solve and visualization
num_sol = damped.numerical_solve(
    initial_conditions={'u': lambda x: np.sin(np.pi * x), 'u_t': lambda x: np.zeros_like(x)},
    t_span=(0.0, 4.0),
    c=1.0,
    beta=0.3,
)
t, u = num_sol.numerical
x = np.linspace(0, 1, u.shape[1])
print(f'c={num_sol.info["c"]}, beta={num_sol.info["beta"]}, CFL={num_sol.info["CFL"]:.4f}')

fig1 = plot_pde_heatmap(x, t, u, title='Damped Wave: $u_{tt} + 2\\beta u_t = c^2 u_{xx}$')
plt.show()

# Show amplitude decay at the midpoint
mid_idx = u.shape[1] // 2
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(t, u[:, mid_idx], 'b-', linewidth=2)
ax.set_xlabel('t')
ax.set_ylabel(f'u(x=0.5, t)')
ax.set_title(f'Damped Wave: Amplitude Decay at Midpoint ($\\beta$={num_sol.info["beta"]})')
ax.grid(True)
plt.tight_layout()
plt.show()